In [ ]:
from pathlib import Path

from plots.wandb_utils import combine_histories, get_wandb_stats

# run_ids = [
#     "7hgw64uc",  # plan storage
#     "rkx580eu",  # basic
#     "4t7jyv2f",  # optim
# ]

run_ids = ["q42zbwis"]


# run_id = "k9xst454"  # tpch - bespoke storage
# run_id = "ggikqms8"  # ceb
# run_id = "761bg8oe"  # ceb - bespoke storage

hists_list = []
for run_id in run_ids:
    summary, hist, config = get_wandb_stats(
        run_id,
        skip_cache=False,  # set to True to skip cache and fetch from W&B API
        wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
    )
    hists_list.append(hist)


hist = combine_histories(hists_list)

/home/jwehrstein/bespoke_olap/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/jwehrstein/.netrc.


✓ Run loaded: tpch_optim1-22v2_wstorage
  State: failed
  Created: 2026-03-12T19:58:26Z
✓ Data fetched: 240 turns, 130 columns
✓ W&B data cached to: /mnt/labstore/bespoke_olap/wandb_cache/b209acabf20a2875e5352285f9bff118aec1c424a4bbee4ace2a6807cedae3fe.pkl
Combined history has 240 rows ([240])


In [2]:
hist

,validation/query_017/speedup,validation/total_duckdb_runtime_ms,validation/query_008/speedup,validation/query_016/duckdb_runtime_ms,validation/query_012/impl_runtime_ms,validation/query_015/speedup,validation/runtime_seconds,validation/query_022/duckdb_runtime_ms,validation/query_009/impl_runtime_ms,tool/shellcommand_count,...,apply_patch/update_count,apply_patch/deleted_loc_count,supervisor,supervisor/approved,validation/compile_error,compile/truncated,compile/error,tool/compile_count,compaction/output_items,compaction/candidate_items
0,0.374615,49703.871575,0.725019,637.923542,815.0,6.559575,555.419423,908.70182,6433.666667,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
236,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN
238,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38,...,NaN,NaN,NaN,NaN,True,NaN,NaN,2.0,NaN,NaN


In [3]:
sorted(list(hist.columns))

['_runtime',
 '_step',
 '_timestamp',
 'agent_name',
 'apply_patch/added_loc_count',
 'apply_patch/deleted_loc_count',
 'apply_patch/update_count',
 'cached_tokens',
 'compaction/candidate_items',
 'compaction/output_items',
 'compile/error',
 'compile/truncated',
 'context_window_usage',
 'cost_usd',
 'current_hash',
 'current_loc',
 'current_prompt',
 'current_prompt_descriptor',
 'input_tokens',
 'llm_hash',
 'output_tokens',
 'prompt_idx',
 'reasoning_tokens',
 'shell/commands',
 'shell/num_commands',
 'supervisor',
 'supervisor/approved',
 'tool/applypatchtool_count',
 'tool/compaction_count',
 'tool/compile_count',
 'tool/handoff_count',
 'tool/llmcall_count',
 'tool/shellcommand_count',
 'tool/validate_count',
 'total/cached_tokens',
 'total/cost_usd',
 'total/input_tokens',
 'total/output_tokens',
 'total/reasoning_tokens',
 'total_runtime',
 'turn',
 'type',
 'validation/all_queries',
 'validation/avg_speedup',
 'validation/compile_error',
 'validation/correct',
 'validation/e

In [ ]:
import re
from typing import Any, Dict, List, Optional

import pandas as pd


def annotate_speedup_over_time(history):
    """
    Track latest per-query runtimes at the largest scale factor and compute
    total speedup (sum duckdb / sum impl) per turn.
    """
    out_col = "validation/largest_sf_total_speedup"
    history[out_col] = None

    runtime_regex = re.compile(
        r"^validation/query_([a-zA-Z0-9]+)/(?P<kind>duckdb_runtime_ms|bespoke_runtime_ms)$"
    )

    # Map query -> runtime column names.
    runtime_cols: Dict[str, Dict[str, str]] = {}
    for col in history.columns:
        m = runtime_regex.match(col)
        if not m:
            continue
        qid = _normalize_query_id(m.group(1))
        runtime_cols.setdefault(qid, {})[m.group("kind")] = col

    if not runtime_cols:
        return out_col

    # Determine expected query set.
    expected_queries = set()
    if "validation/query_ids_executed" in history.columns:
        for qids in history["validation/query_ids_executed"]:
            if isinstance(qids, list):
                expected_queries.update(_normalize_query_id(qid) for qid in qids)

    # Fallback for runs without query_ids_executed in history.
    available_query_pairs = {
        qid
        for qid, kinds in runtime_cols.items()
        if "duckdb_runtime_ms" in kinds and "bespoke_runtime_ms" in kinds
    }
    if expected_queries:
        expected_queries = expected_queries.intersection(available_query_pairs)
    else:
        # Fallback for runs without query_ids_executed in history.
        expected_queries = available_query_pairs

    if not expected_queries:
        return out_col

    # Largest sf present in this history.
    max_sf = None
    if "validation/scale_factor" in history.columns:
        sf_values = pd.to_numeric(
            history["validation/scale_factor"], errors="coerce"
        ).dropna()
        if len(sf_values) > 0:
            max_sf = float(sf_values.max())

    # Track the latest runtime per query (at max_sf only when available).
    current_runtimes: Dict[str, Dict[str, float]] = {}
    speedup_series: List[Optional[float]] = []

    for _, row in history.iterrows():
        row_sf = pd.to_numeric(row.get("validation/scale_factor"), errors="coerce")  # type: ignore
        sf_matches = max_sf is None or (
            pd.notna(row_sf) and abs(float(row_sf) - float(max_sf)) < 1e-12
        )

        if sf_matches:
            for qid in expected_queries:
                cols_for_query = runtime_cols.get(qid, {})
                impl_col = cols_for_query.get("bespoke_runtime_ms")
                duck_col = cols_for_query.get("duckdb_runtime_ms")
                if impl_col is None or duck_col is None:
                    continue

                impl_val = row.get(impl_col)
                duck_val = row.get(duck_col)

                if pd.notna(impl_val):
                    current_runtimes.setdefault(qid, {})["bespoke_runtime_ms"] = float(
                        impl_val
                    )
                if pd.notna(duck_val):
                    current_runtimes.setdefault(qid, {})["duckdb_runtime_ms"] = float(
                        duck_val
                    )

        have_all = all(
            qid in current_runtimes
            and "bespoke_runtime_ms" in current_runtimes[qid]
            and "duckdb_runtime_ms" in current_runtimes[qid]
            for qid in expected_queries
        )

        if not have_all:
            speedup_series.append(None)
            continue

        total_impl = sum(
            current_runtimes[qid]["bespoke_runtime_ms"] for qid in expected_queries
        )
        total_duck = sum(
            current_runtimes[qid]["duckdb_runtime_ms"] for qid in expected_queries
        )

        if total_impl <= 0:
            speedup_series.append(None)
        else:
            speedup_series.append(total_duck / total_impl)

    history[out_col] = speedup_series


def _normalize_query_id(raw_query_id: Any) -> str:
    """Normalize query ids so '020' and '20' map to the same key."""
    q = str(raw_query_id)
    if q.isdigit():
        return str(int(q))
    return q


annotate_speedup_over_time(hist)

In [ ]:
df2 = hist[
    [
        "_step",
        "validation/largest_sf_total_speedup",
        "validation/scale_factor",
        "validation/num_queries",
        "validation/external_call",
        "validation/compile_with_optimize",
        "validation/instantiations",
        "validation/repetitions",
        "validation/trace_mode",
    ]
].dropna()
df2

,_step,validation/largest_sf_total_speedup,validation/scale_factor,validation/num_queries,validation/external_call,validation/fasttest_optimize,validation/instantiations,validation/repetitions,validation/trace_mode
0,0,0.808228,20.0,22.0,True,True,22.0,3.0,False
30,30,0.731930,20.0,3.0,False,True,3.0,1.0,True
35,35,0.812301,20.0,3.0,True,True,3.0,3.0,False
36,36,0.731876,20.0,3.0,True,True,3.0,1.0,True
55,55,0.683725,20.0,3.0,False,True,3.0,1.0,True
58,58,0.731617,20.0,3.0,True,True,3.0,3.0,False
59,59,0.683443,20.0,3.0,True,True,3.0,1.0,True
72,72,0.672637,20.0,3.0,False,True,3.0,1.0,True
80,80,0.664624,20.0,3.0,False,True,3.0,1.0,True
83,83,0.684453,20.0,3.0,True,True,3.0,3.0,False
